# 02. Protocole de validation, modèle de référence et SARIMA

**Protocole commun à tous les modèles** (défini dans `src/models/validation.py`) :
* Entraînement jusqu'au 8 décembre 23 h (heure de Milan), puis deux fenêtres de test de 7 jours (9 au 15 et 16 au 22 décembre), chacune réentraînée sur tout l'historique précédent.
* Test de résistance à part : du 23 décembre au 1er janvier (jours fériés, forte baisse du niveau).
* Métrique principale : MAE. Compléments : RMSE, MAPE, et MAE sur les heures chargées (jours ouvrés non fériés, 12 h à 18 h).
* Les anomalies ne sont pas supprimées : les jours fériés connus (1er nov., 25 et 26 déc., 1er janv.) sont fournis aux modèles comme variable.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.validation import WINDOWS, STRESS, make_window, evaluate, TZ
from src.models.baseline import naive_seasonal
from src.models.sarima import fit_sarima, sarima_forecast

sns.set_theme(style="whitegrid")
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

s = pd.read_csv(ROOT / "data" / "processed" / "traffic_zone_1h.csv", index_col=0)
s.index = pd.to_datetime(s.index, utc=True).tz_convert(TZ)
s = s["internet"].asfreq("1h")

ALL = {**WINDOWS, **STRESS}
for name, (a, b) in ALL.items():
    tr, te = make_window(s, a, b)
    print(f"{name}: {len(tr)} h d'entraînement, {len(te)} h de test")

## 1. Modèles de référence
Naïf 24 h (répète la veille) et naïf 168 h (répète la semaine précédente). Le second est la référence honnête pour un horizon de 7 jours.

In [ ]:
results = []
results.append(evaluate(lambda tr, ix: naive_seasonal(tr, ix, 24), s, ALL, "Naïf 24 h"))
results.append(evaluate(lambda tr, ix: naive_seasonal(tr, ix, 168), s, ALL, "Naïf 168 h"))
pd.concat(results).round(0)

## 2. SARIMA
Choix de départ issus de l'exploration (à confirmer par l'AIC et les résidus) :
* partie saisonnière de période 24 avec différenciation saisonnière (D=1), car la série n'est stationnaire qu'après différenciation à 24 h ;
* partie autorégressive d'ordre 2 (PACF : pics aux retards 1 et 2) ;
* variables exogènes week-end et jour férié pour la saisonnalité hebdomadaire (période 168 trop lourde pour ~900 points) ;
* transformation logarithmique, car l'amplitude du cycle journalier diminue avec le niveau. On compare avec la version sans log.

In [ ]:
res_sarima = []
res_sarima.append(evaluate(lambda tr, ix: sarima_forecast(tr, ix, log=False), s, ALL, "SARIMA (brut)"))
res_sarima.append(evaluate(lambda tr, ix: sarima_forecast(tr, ix, log=True), s, ALL, "SARIMA (log)"))
results += res_sarima
table = pd.concat(results, ignore_index=True)
table.round(0)

## 3. Comparaison

In [ ]:
piv = table.pivot(index="modèle", columns="fenêtre", values="MAE").round(0)
display(piv)
ax = piv.T.plot(kind="bar", figsize=(11, 4), rot=15)
ax.set_ylabel("MAE (unité relative)"); ax.set_title("MAE par modèle et par fenêtre de test")
plt.tight_layout(); plt.savefig(FIG / "06_mae_modeles_e2.png", dpi=150)
table.to_csv(ROOT / "reports" / "resultats_e2.csv", index=False)

In [ ]:
best = "SARIMA (log)"
fig, axes = plt.subplots(len(ALL), 1, figsize=(12, 3.2 * len(ALL)))
for ax, (name, (a, b)) in zip(axes, ALL.items()):
    tr, te = make_window(s, a, b)
    ax.plot(te.index, te, label="réel", lw=1.2)
    ax.plot(te.index, naive_seasonal(tr, te.index, 168), label="naïf 168 h", lw=1, alpha=0.8)
    ax.plot(te.index, sarima_forecast(tr, te.index, log=True), label=best, lw=1, alpha=0.9)
    ax.set_title(name); ax.legend(loc="upper right", ncol=3)
plt.tight_layout(); plt.savefig(FIG / "07_previsions_e2.png", dpi=150)

## 4. Diagnostic des résidus (ajustement sur l'entraînement initial)

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
train0, _ = make_window(s, *WINDOWS[list(WINDOWS)[0]])
fit = fit_sarima(train0, log=True)
print(fit.summary().tables[1])
resid = fit.resid.iloc[24:]
print(acorr_ljungbox(resid, lags=[24, 48, 168], return_df=True))
fig = fit.plot_diagnostics(figsize=(12, 7), lags=72); fig.tight_layout()
fig.savefig(FIG / "08_diagnostic_sarima.png", dpi=150)

## 5. Conclusions du livrable E2
À compléter après exécution sur les données réelles :
* Le SARIMA bat-il le naïf 168 h ? De combien (MAE, MAE en heures chargées) ?
* Effet de la transformation logarithmique
* Comportement sur le test de résistance (jours fériés, baisse de niveau) et limites : un seul Noël dans les données, effet « jour férié » estimé sur le 1er novembre seulement
* Résidus : autocorrélation restante (test de Ljung-Box)